[Reference](https://python.plainenglish.io/9-fastapi-refactors-that-make-legacy-apis-feel-like-new-db1a3545bec5)

# 1. Move Business Logic Out of Endpoints

In [2]:
# services/user_service.py
class UserService:
    def create_user(self, user_data):
        # business rules go here
        pass

# routes/users.py
@router.post("/users")
def register_user(user: UserIn, service: UserService = Depends()):
    return service.create_user(user)

# 2. Switch to Pydantic for All Request & Response Models

In [3]:
from pydantic import BaseModel

class UserCreateRequest(BaseModel):
    username: str
    email: str

# in your endpoint
@app.post("/signup", response_model=UserReadResponse)
def signup(request: UserCreateRequest):
    ...

# 3. Adopt Dependency Injection Everywhere
If you’re reaching for globals or singletons, switch to FastAPI’s `Depends()` pattern for injecting configuration, DB sessions, and services. It makes your app easier to test and change.

# 4. Use Custom Exception Handlers

In [4]:
@app.exception_handler(MyAppError)
async def my_error_handler(request, exc):
    return JSONResponse(
        status_code=exc.status_code,
        content={"detail": exc.message}
    )

# 5. Integrate Structured Logging with Loguru

In [5]:
from loguru import logger

logger.add("app.log", rotation="10 MB", enqueue=True, backtrace=True)

# 6. Split Large Files and Projects into Submodules
```
project/
├── api/
│   ├── users.py
│   └── items.py
├── services/
│   └── user_service.py
├── models/
└── main.py
```

# 7. Use Async Everywhere

In [6]:
@app.get("/external")
async def read_external():
    async with httpx.AsyncClient() as client:
        r = await client.get("https://example.com")
        return r.json()

# 8. Centralize Configuration Using Environment Variables
Hardcoded API keys and URLs make deployments risky. Use a config class (with Pydantic’s `BaseSettings`) that reads from .env files or the environment for safe, consistent configuration.

# 9. Add Health, Metrics, and Readiness Endpoints
Long-running projects need observability. Include routes for /health checks and readiness, and consider exposing Prometheus metrics. This pays off as your team grows and your code runs in production.